# My Viva Practice: Predict and Explain

Run the cells in order. Start with Linear Regression, then switch KIND to forest and run all again. These cells use the trained models; they do not repeat fitting or tuning. Keep the existing data/preprocessing limitations in your explanation. Choose the FDM Project .venv Python kernel.

## 1. Load the tools

**Say:** Pandas handles tables, NumPy handles numerical arrays, joblib loads the trained model, and scikit-learn provides the error calculations. The path code finds our project so its model classes can be imported.

In [1]:
from pathlib import Path
import sys
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from IPython.display import display

start = Path.cwd()
ROOT = next(p for p in [start, *start.parents] if (p / "src" / "advanced_training.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from viva.live_demo import load_inputs, verify_predictions
from src.regression_common import prediction_table
print("Project:", ROOT)

Project: /Users/chelakawijesekara/Downloads/FDM Project


## 2. Choose the model and load the inputs

**Say:** X contains the input features, and y contains the actual energy readings. The helper checks that the prepared data matches our recorded experiment. We are not fitting a model here. Change KIND to forest to demonstrate Random Forest.

In [2]:
KIND = "linear"  # Change to "forest" for Random Forest.
ARTIFACTS = None  # Keep None here; if data was moved, set its original prepared-data folder.
output_dir, metadata, X_train, X_test, y_train, y_test = load_inputs(KIND, ARTIFACTS)
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Input features:", X_test.shape[1])
print("Test dates:", X_test.index.min(), "to", X_test.index.max())

Training rows: 28511
Test rows: 5857
Input features: 15
Test dates: 2018-11-01 00:00:00 to 2019-01-01 00:00:00


## 3. Look at the inputs and answers

**Say:** Each row is one 15-minute interval. For example, lag_1 is the previous actual usage and lag_96 is the usage one day earlier. head displays five rows; it does not train anything.

In [3]:
display(X_test.head())
display(y_test.head())

,tod_sin,tod_cos,dow_sin,dow_cos,is_weekend,load_level,lag_1,lag_4,lag_96,lag_672,roll_mean_4,roll_std_4,roll_mean_96,roll_max_96,diff_1
timestamp,,,,,,,,,,,,,,,
2018-11-01 00:00:00,0.000000,1.000000,0.433884,-0.900969,0,0,3.92,3.96,3.67,2.92,3.9150,0.033166,41.465104,139.43,0.03
2018-11-01 00:15:00,0.065403,0.997859,0.433884,-0.900969,0,0,3.85,3.89,3.64,3.10,3.8875,0.028723,41.466979,139.43,-0.07
2018-11-01 00:30:00,0.130526,0.991445,0.433884,-0.900969,0,0,3.89,3.89,3.60,3.67,3.8875,0.028723,41.469583,139.43,0.04
2018-11-01 00:45:00,0.195090,0.980785,0.433884,-0.900969,0,0,3.96,3.92,3.67,3.02,3.9050,0.046547,41.473333,139.43,0.07
2018-11-01 01:00:00,0.258819,0.965926,0.433884,-0.900969,0,0,5.11,3.85,3.56,2.99,4.2025,0.606706,41.488333,139.43,1.15


timestamp
2018-11-01 00:00:00    3.85
2018-11-01 00:15:00    3.89
2018-11-01 00:30:00    3.96
2018-11-01 00:45:00    5.11
2018-11-01 01:00:00    4.72
Name: Usage_kWh, dtype: float64

## 4. Load the trained model and predict

**Say:** joblib.load restores the fitted estimator. predict receives X_test, not y_test. Its saved preprocessing is applied internally. This forest automatically adds its predicted change to previous usage; I must not add it twice. Only load our trusted project artifact.

In [4]:
model = joblib.load(output_dir / "model.joblib")
predictions = model.predict(X_test)
print("Predictions generated:", len(predictions))
print("First five predictions:", predictions[:5])

Predictions generated: 5857
First five predictions: [4.38285641 4.37873132 4.53685883 4.53409071 6.38306344]


## 5. Type the metric calculation

**Say:** MAE averages absolute errors. RMSE gives more weight to large errors. R2 is not percentage accuracy. The mask selects actual usage of at least 50 kWh for the high-load error. The answers are used for evaluation only after prediction.

In [5]:
HIGH_LOAD_KWH = 50.0

def calculate_metrics(actual, predicted):
    actual = np.asarray(actual)
    predicted = np.asarray(predicted)
    mask = actual >= HIGH_LOAD_KWH
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "R2": r2_score(actual, predicted),
        "HighLoad_MAE": mean_absolute_error(actual[mask], predicted[mask]) if mask.any() else np.nan,
    }

print("High-load test intervals:", int((y_test >= HIGH_LOAD_KWH).sum()))

High-load test intervals: 1491


## 6. Compare with the naive prediction

**Say:** The naive rule copies the previous usage. The table scores the naive and learned predictions on exactly the same rows. Lower error is better. This is a test table, not the cross-validation table used to select settings.

In [6]:
naive_predictions = X_test["lag_1"].to_numpy()
results = pd.DataFrame({
    "Naive (last interval)": calculate_metrics(y_test, naive_predictions),
    metadata["settings"]["algorithm"]: calculate_metrics(y_test, predictions),
}).T
display(results.round(3))

examples = prediction_table(y_test, predictions, X_test["lag_1"])
display(examples.head().round(3))

,MAE,RMSE,R2,HighLoad_MAE
Naive (last interval),5.155,11.813,0.856,12.420
Linear Regression,4.128,7.880,0.936,9.671


,actual_kWh,predicted_kWh,persistence_kWh,residual_kWh
timestamp,,,,
2018-11-01 00:00:00,3.85,4.383,3.92,-0.533
2018-11-01 00:15:00,3.89,4.379,3.85,-0.489
2018-11-01 00:30:00,3.96,4.537,3.89,-0.577
2018-11-01 00:45:00,5.11,4.534,3.96,0.576
2018-11-01 01:00:00,4.72,6.383,5.11,-1.663


## 7. Verify this is the recorded model

**Say:** I am checking that these newly generated predictions match our saved experiment. Passing this check confirms reproducibility on these inputs; it does not prove there is no leakage or that future errors will be identical.

In [7]:
verify_predictions(output_dir, examples)
print("PASS: predictions, target values and timestamps match the saved experiment.")

PASS: predictions, target values and timestamps match the saved experiment.


## 8. Explain how settings were selected

**Say:** Settings were selected by the lowest mean RMSE across five time-series validation folds. High-load MAE was also reported, but it was not the selection objective. These cells only display that evidence.

In [8]:
print(json.dumps(metadata["settings"]["selected_specification"], indent=2))
print("Selection:", metadata["selection_metric"])
print("Mean CV RMSE:", round(metadata["settings"]["cv_RMSE_kWh"], 3))
display(pd.read_csv(output_dir / "fold_boundaries.csv"))
for note in metadata["limitations"]:
    print("-", note)

{
  "knots": 5,
  "schedule": "weekend",
  "use_load": false
}
Selection: mean of five TimeSeriesSplit fold RMSE values; lowest wins
Mean CV RMSE: 9.912


,fold,fit_rows,validation_rows,fit_start,fit_end,validation_start,validation_end
0,1,4756,4751,2018-01-08 00:15:00,2018-02-26 13:00:00,2018-02-26 13:15:00,2018-04-17 00:45:00
1,2,9507,4751,2018-01-08 00:15:00,2018-04-17 00:45:00,2018-04-17 01:00:00,2018-06-05 12:30:00
2,3,14258,4751,2018-01-08 00:15:00,2018-06-05 12:30:00,2018-06-05 12:45:00,2018-07-25 00:15:00
3,4,19009,4751,2018-01-08 00:15:00,2018-07-25 00:15:00,2018-07-25 00:30:00,2018-09-12 12:00:00
4,5,23760,4751,2018-01-08 00:15:00,2018-09-12 12:00:00,2018-09-12 12:15:00,2018-10-31 23:45:00


- Forecasts are one interval ahead (15 minutes), with previous actual readings available before each prediction. This is not a day-ahead test.
- The selected transformer ignores load_level. It remains in the 15-column input schema, but no upcoming operating-load category is required.
- The supplied preprocessing shifts midnight rows to the next day; the original interval timestamp convention remains an assumption.
- The supplied preprocessing interpolates the target at 2018-11-08 00:00 using a future reading. This affects that test target and later lag/rolling features. Test results describe those supplied artifacts, not a fully causal evaluation of raw measurements.
- Upstream feature selection used the entire supplied training period, including September/October. Validation is provisional conditional on that fixed feature set; strict validation needs feature selection refitted on the earlier fitting period.
- The preprocessing notebook already inspected the test persistence score and

## Optional: demonstrate fitting only when asked

The following is intentionally a text example, so Run All does not retrain. Type it in a NEW code cell only to practise fitting. Start with KIND = "linear". The forest can take several minutes. It refits the selected configuration in memory, does not repeat the search, and does not overwrite your official model.

```python
from src.advanced_training import make_model
practice_model = make_model(KIND, metadata["settings"]["selected_specification"], jobs=2)
practice_model.fit(X_train, y_train)
practice_predictions = practice_model.predict(X_test)
display(pd.DataFrame([calculate_metrics(y_test, practice_predictions)]).round(3))
```

**Say:** fit learns from the training inputs and answers. predict then uses test inputs only. Evaluation compares the predictions with the test answers afterwards. Do not tune repeatedly based on this test result.

## Final selection answer

Random Forest is a leading candidate, not a verified winner across all five. KNN and Gradient Boosting used different test dates and high-load thresholds. First align the evaluation, choose the primary metric on validation data, and acknowledge that existing test results have already been seen. Read START_HERE.md and VIVA_GUIDE.md beside this notebook.